## 1. Load the Dataset

The PhiUSIIL Phishing URL Dataset is obtained from the UCI Machine Learning
Repository. It contains website and URL-related features that can be used to
classify websites as legitimate or phishing.

The raw dataset is loaded without modifying the original data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Get the raw data directory
raw_dir = Path("../data/raw")

# Get the CSV file
csv_file = list(raw_dir.glob("*.csv"))[0]

# Load the dataset
df = pd.read_csv(csv_file)

print("Dataset loaded successfully.")
print(df.head())

Dataset loaded successfully.
     FILENAME                                 URL  URLLength  \
0  521848.txt    https://www.southbankmosaics.com         31   
1   31372.txt            https://www.uni-mainz.de         23   
2  597387.txt      https://www.voicefmradio.co.uk         29   
3  554095.txt         https://www.sfnmjournal.com         26   
4  151578.txt  https://www.rewildingargentina.org         33   

                       Domain  DomainLength  IsDomainIP  TLD  \
0    www.southbankmosaics.com            24           0  com   
1            www.uni-mainz.de            16           0   de   
2      www.voicefmradio.co.uk            22           0   uk   
3         www.sfnmjournal.com            19           0  com   
4  www.rewildingargentina.org            26           0  org   

   URLSimilarityIndex  CharContinuationRate  TLDLegitimateProb  ...  Pay  \
0               100.0              1.000000           0.522907  ...    0   
1               100.0              0.666667      

In [3]:
df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


## 2. Dataset Dimensions

First, we inspect the number of observations and columns in the dataset.

In [4]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 235795
Number of columns: 56


In [5]:
print("Dataset columns:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

Dataset columns:
 1. FILENAME
 2. URL
 3. URLLength
 4. Domain
 5. DomainLength
 6. IsDomainIP
 7. TLD
 8. URLSimilarityIndex
 9. CharContinuationRate
10. TLDLegitimateProb
11. URLCharProb
12. TLDLength
13. NoOfSubDomain
14. HasObfuscation
15. NoOfObfuscatedChar
16. ObfuscationRatio
17. NoOfLettersInURL
18. LetterRatioInURL
19. NoOfDegitsInURL
20. DegitRatioInURL
21. NoOfEqualsInURL
22. NoOfQMarkInURL
23. NoOfAmpersandInURL
24. NoOfOtherSpecialCharsInURL
25. SpacialCharRatioInURL
26. IsHTTPS
27. LineOfCode
28. LargestLineLength
29. HasTitle
30. Title
31. DomainTitleMatchScore
32. URLTitleMatchScore
33. HasFavicon
34. Robots
35. IsResponsive
36. NoOfURLRedirect
37. NoOfSelfRedirect
38. HasDescription
39. NoOfPopup
40. NoOfiFrame
41. HasExternalFormSubmit
42. HasSocialNet
43. HasSubmitButton
44. HasHiddenFields
45. HasPasswordField
46. Bank
47. Pay
48. Crypto
49. HasCopyrightInfo
50. NoOfImage
51. NoOfCSS
52. NoOfJS
53. NoOfSelfRef
54. NoOfEmptyRef
55. NoOfExternalRef
56. label


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 56 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   FILENAME                    235795 non-null  object 
 1   URL                         235795 non-null  object 
 2   URLLength                   235795 non-null  int64  
 3   Domain                      235795 non-null  object 
 4   DomainLength                235795 non-null  int64  
 5   IsDomainIP                  235795 non-null  int64  
 6   TLD                         235795 non-null  object 
 7   URLSimilarityIndex          235795 non-null  float64
 8   CharContinuationRate        235795 non-null  float64
 9   TLDLegitimateProb           235795 non-null  float64
 10  URLCharProb                 235795 non-null  float64
 11  TLDLength                   235795 non-null  int64  
 12  NoOfSubDomain               235795 non-null  int64  
 13  HasObfuscation

In [9]:
missing_values = df.isnull().sum()

print("Columns containing missing values:")
print(missing_values[missing_values > 0])

Columns containing missing values:
Series([], dtype: int64)


In [7]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [8]:
df["label"].value_counts()
df["label"].value_counts(normalize=True) * 100

label
1    57.189508
0    42.810492
Name: proportion, dtype: float64

## 3. Categorical and Text-Based Features

The dataset contains five object-type columns: `FILENAME`, `URL`, `Domain`,
`TLD`, and `Title`.

These columns require additional investigation before the final feature
selection and preprocessing strategy is defined.

In [10]:
object_columns = df.select_dtypes(include="object").columns

print("Object-type columns:")
for column in object_columns:
    print(f"- {column}")

Object-type columns:
- FILENAME
- URL
- Domain
- TLD
- Title


In [11]:
for column in object_columns:
    print(f"{column}: {df[column].nunique():,} unique values")

FILENAME: 235,795 unique values
URL: 235,370 unique values
Domain: 220,086 unique values
TLD: 695 unique values
Title: 197,874 unique values


In [12]:
print("Number of unique TLDs:", df["TLD"].nunique())

print("\nMost common TLDs:")
print(df["TLD"].value_counts().head(20))

Number of unique TLDs: 695

Most common TLDs:
TLD
com     112554
org      18793
net       7097
app       6508
uk        6395
co        5422
io        4201
de        3996
ru        3875
au        2979
dev       2345
top       2329
jp        2219
it        1887
edu       1861
fr        1858
br        1846
nl        1727
ca        1614
info      1566
Name: count, dtype: int64


In [13]:
print("Unique URLs:", df["URL"].nunique())
print("Unique Domains:", df["Domain"].nunique())

Unique URLs: 235370
Unique Domains: 220086


In [14]:
print("Duplicate URLs:", df["URL"].duplicated().sum())
print("Duplicate Domains:", df["Domain"].duplicated().sum())

Duplicate URLs: 425
Duplicate Domains: 15709


In [15]:
print("Unique titles:", df["Title"].nunique())

Unique titles: 197874


In [16]:
df["Title"].head(10)

0    à¸‚à¹ˆà¸²à¸§à¸ªà¸” à¸‚à¹ˆà¸²à¸§à¸§à¸±à¸™à¸™à¸µ...
1                johannes gutenberg-universitÃ¤t mainz
2                                 voice fm southampton
3    home page: seminars in fetal and neonatal medi...
4                       fundaciÃ³n rewilding argentina
5                                           gri - home
6                                                    0
7                                          nerds candy
8    hyderabadonline - business listing in hyderaba...
9                                                 home
Name: Title, dtype: object

## 4. Descriptive Statistics of Numerical Features

Descriptive statistics are used to examine the distribution, scale and
variability of numerical features before preprocessing.